# GUS01D: GeoTERYT Geometry Assignment Methods

This notebook demonstrates the geometry assignment methods in GeoTERYT database v3.1.

## Overview
- Load complete database from GUS01C
- Demonstrate geometry assignment workflow:
  1. `assign_geometries()` - exact match for same-year teryt_ids
  2. `assign_missing_geometries()` - find geometries for unchanged units
  3. `impute_geometries_past_tid()` - use historical codes for imputation
  4. `impute_from_best_candidates()` - apply best geometry candidates
  5. `country_shape_check()` - verify coverage against Poland boundary

In [ ]:
import os
import sys
import pandas as pd
from pathlib import Path
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import importlib

# Change root directory to the repo root
def find_repo_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
            return p
    return start

repo_root = find_repo_root()
geospatial_root = repo_root.parent.parent / 'Data' / 'Geospatial'

os.chdir(repo_root)
if str(repo_root / 'Code' / 'tools') not in sys.path:
    sys.path.append(str(repo_root / 'Code' / 'tools'))

# Import local toolkit
import geoTERYT_db as gtdb
importlib.reload(gtdb)

print(f"Repo root: {repo_root}")
print(f"geoTERYT_db version: {gtdb.__doc__.split(chr(10))[0] if gtdb.__doc__ else 'unknown'}")

## 1. Load the Complete Database

Load the database saved from GUS01C (includes harmonized data + stored geometries).

In [ ]:
# Load complete database
complete_db_path = geospatial_root / 'geoteryt_complete.pkl'
db = gtdb.load_complete_database(complete_db_path)

# Print summary
db.print_summary()

## 2. Check Initial Geometry Status

Before assignment, check how many records have geometries.

In [ ]:
# Check geometry status for different years
years_to_check = [1999, 2005, 2010, 2015, 2020, 2024]

for year in years_to_check:
    snapshot = db.get_snapshot(year)
    gminas = [r for r in snapshot if r.level == 6 and r.rodz not in ['4', '5', '8', '9']]
    with_geom = sum(1 for r in gminas if r.has_geometry)
    print(f"{year}: {with_geom}/{len(gminas)} gminas with geometry ({with_geom/len(gminas)*100:.1f}%)")

In [ ]:
# Check which geometry years are available
print(f"Geometry years available: {sorted(db._geometries.keys())}")
for year, gdf in sorted(db._geometries.items()):
    print(f"  {year}: {len(gdf)} geometries")

## 3. Assign Geometries - Exact Match

Use `assign_geometries(year, level)` to assign geometries where teryt_id matches exactly.

In [ ]:
# Assign geometries for year 2024 (most recent)
result = db.assign_geometries(year=2024, level=6, verbose=True)

In [ ]:
# Assign for other available years
for year in sorted(db._geometries.keys(), reverse=True):
    print(f"\n=== Year {year} ===")
    result = db.assign_geometries(year=year, level=6, verbose=True)

## 4. Assign Missing Geometries - Cross-Year Matching

For units without geometry that haven't changed, find matching geometries from other years.

In [ ]:
# Try to find geometries for 2024 units that are missing
result = db.assign_missing_geometries(year=2024, level=6, tolerance=0.1, verbose=True)

In [ ]:
# Check status after assignment
year = 2024
snapshot = db.get_snapshot(year)
gminas = [r for r in snapshot if r.level == 6 and r.rodz not in ['4', '5', '8', '9']]
with_geom = sum(1 for r in gminas if r.has_geometry)
print(f"{year}: {with_geom}/{len(gminas)} gminas with geometry ({with_geom/len(gminas)*100:.1f}%)")

# Show some that are still missing
missing = [r for r in gminas if not r.has_geometry][:5]
print(f"\nSample missing gminas:")
for r in missing:
    print(f"  {r.teryt_id} - {r.name} (has_changes={r.has_changes}, historical_codes={r.historical_codes[:3] if r.historical_codes else []}...)")

## 5. Impute Using Historical Codes

For units with changes, use `code_by_year` to find affiliated teryt_ids and search for geometry candidates.

In [ ]:
# Check a unit with historical codes
sample = None
for r in db._records.values():
    if r.historical_codes and len(r.historical_codes) > 1 and r.level == 6:
        sample = r
        break

if sample:
    print(f"Sample unit: {sample.teryt_id} - {sample.name}")
    print(f"  historical_codes: {sample.historical_codes}")
    print(f"  code_by_year: {sample.code_by_year}")
    print(f"  has_geometry: {sample.has_geometry}")

In [ ]:
# Impute geometries using historical codes
result = db.impute_geometries_past_tid(year=2024, level=6, verbose=True)

In [ ]:
# Apply best candidates
result = db.impute_from_best_candidates(verbose=True)

## 6. Country Shape Check

Verify geometry coverage by overlaying with Poland boundary.

In [ ]:
# Check coverage for different years
for year in [1999, 2010, 2020, 2024]:
    print(f"\n=== Year {year} ===")
    result = db.country_shape_check(year=year, level=6, verbose=True)

## 7. Visualize Coverage

In [ ]:
# Visualize coverage for a specific year
year = 2024

gdf = db.to_geodataframe(year=year, level=6, exclude_subdivisions=True, only_with_geometry=True)
poland = db.get_poland_boundary()

fig, ax = plt.subplots(1, 1, figsize=(12, 12))

# Plot Poland boundary first (as background)
if poland is not None:
    gpd.GeoSeries([poland]).plot(ax=ax, color='lightcoral', alpha=0.5, label='Missing')

# Plot gminas with geometry
gdf.plot(ax=ax, facecolor='lightgreen', edgecolor='darkgreen', linewidth=0.1, alpha=0.8)

ax.set_title(f'Geometry Coverage for Year {year} ({len(gdf)} gminas with geometry)')
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Show gminas missing geometry (if any) with geometry_notes
year = 2024
snapshot = db.get_snapshot(year)
gminas = [r for r in snapshot if r.level == 6 and r.rodz not in ['4', '5', '8', '9']]
missing = [r for r in gminas if not r.has_geometry]

print(f"Gminas missing geometry in {year}: {len(missing)}")
if missing:
    for r in missing[:20]:
        print(f"  {r.teryt_id} - {r.name} (notes: {r.geometry_notes})")

## 8. Save Updated Database

Save the database with assigned geometries.

In [ ]:
# Save the updated database
output_path = geospatial_root / 'geoteryt_complete_v31.pkl'
db.save_complete(output_path, verbose=True)

## Summary

The geometry assignment workflow in v3.1:

1. **`load_geometries()`** - Stores geometry files in `_geometries` dict (from GUS01C)
2. **`assign_geometries(year, level)`** - Assigns geometry where teryt_id matches exactly
3. **`assign_missing_geometries(year, level)`** - For unchanged units, finds geometry from other years
4. **`impute_geometries_past_tid(year, level)`** - Uses `code_by_year` to find affiliated teryt_ids
5. **`impute_from_best_candidates()`** - Applies geometry_best_candidate to records
6. **`country_shape_check(year, level)`** - Verifies coverage against Poland boundary

Key advantages:
- Explicit control over geometry assignment
- Clear tracking via `geometry_notes`
- Better handling of units with ID changes
- Verification against national boundary